In [1]:
import os
import pathlib
import sys
import time

import pandas as pd
import psutil
import tomli
from image_analysis_3D.file_utils.arg_parsing_utils import (
    check_for_missing_args,
    parse_args,
)
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()

from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
    save_features_as_parquet,
)
from image_analysis_3D.featurization_utils.loading_classes import (
    ImageSetLoader,
    ObjectLoader,
)
from image_analysis_3D.featurization_utils.resource_profiling_util import (
    start_profiling,
    stop_profiling,
)
from image_analysis_3D.featurization_utils.texture_utils import measure_3D_texture

profile_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot/NF1_organoid_data")).resolve(),
    root_dir,
)

In [2]:
if not in_notebook:
    arguments_dict = parse_args()
    patient = arguments_dict["patient"]
    well_fov = arguments_dict["well_fov"]
    channel = arguments_dict["channel"]
    compartment = arguments_dict["compartment"]
    processor_type = arguments_dict["processor_type"]
    input_subparent_name = arguments_dict["input_subparent_name"]
    mask_subparent_name = arguments_dict["mask_subparent_name"]
    output_features_subparent_name = arguments_dict["output_features_subparent_name"]

else:
    well_fov = "C4-1"
    patient = "NF0014_T1"
    channel = "Mito"
    compartment = "Nuclei"
    processor_type = "CPU"
    input_subparent_name = "zstack_images"
    mask_subparent_name = "segmentation_masks"
    output_features_subparent_name = "extracted_features"

image_set_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{input_subparent_name}/{well_fov}/"
)
mask_set_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{mask_subparent_name}/{well_fov}/"
)
output_parent_path = pathlib.Path(
    f"{profile_base_dir}/data/{patient}/{output_features_subparent_name}/{well_fov}/"
)
output_parent_path.mkdir(parents=True, exist_ok=True)
channel_mapping_file_path = pathlib.Path(
    f"{root_dir}/config/channel_mapping.toml"
).resolve(strict=True)

In [3]:
# read in channel mapping
with open(channel_mapping_file_path, "rb") as f:
    channel_mapping_dict = tomli.load(f)
channel_n_compartment_mapping = channel_mapping_dict["channel_mapping"]

In [4]:
start_time, start_mem = start_profiling()

In [5]:
image_set_loader = ImageSetLoader(
    image_set_path=image_set_path,
    mask_set_path=mask_set_path,
    anisotropy_spacing=(1, 0.1, 0.1),
    channel_mapping=channel_n_compartment_mapping,
    image_set_name=well_fov,
    mask_key_name=[channel_n_compartment_mapping[compartment]],
    raw_image_key_name=[channel_n_compartment_mapping[channel]],
)

In [6]:
object_loader = ObjectLoader(
    image_set_loader.image_set_dict[channel],
    image_set_loader.image_set_dict[compartment],
    channel,
    compartment,
)
output_texture_dict = measure_3D_texture(
    object_loader=object_loader,
    distance=3,  # distance in pixels 3 is what CP uses
)
final_df = pd.DataFrame(output_texture_dict)

final_df = final_df.pivot(
    index="object_id",
    columns="texture_name",
    values="texture_value",
)
final_df.reset_index(inplace=True)
final_df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment=compartment,
            channel=channel,
            feature_type="Texture",
            measurement=col,
        )
        if col != "object_id"
        else col
        for col in final_df.columns
    },
    inplace=True,
)
final_df.insert(0, "image_set", image_set_loader.image_set_name)
final_df.columns.name = None

save_path = save_features_as_parquet(
    parent_path=output_parent_path,
    df=final_df,
    feature_type="Texture",
    channel=channel,
    compartment=compartment,
    cpu_or_gpu=processor_type,
)
final_df.head()

55it [06:23,  6.98s/it]


,image_set,object_id,Nuclei_Mito_Texture_AngularSecondMoment-256-3,Nuclei_Mito_Texture_Contrast-256-3,Nuclei_Mito_Texture_Correlation-256-3,Nuclei_Mito_Texture_DifferenceEntropy-256-3,Nuclei_Mito_Texture_DifferenceVariance-256-3,Nuclei_Mito_Texture_Entropy-256-3,Nuclei_Mito_Texture_InformationMeasureOfCorrelation1-256-3,Nuclei_Mito_Texture_InformationMeasureOfCorrelation2-256-3,Nuclei_Mito_Texture_InverseDifferenceMoment-256-3,Nuclei_Mito_Texture_SumAverage-256-3,Nuclei_Mito_Texture_SumEntropy-256-3,Nuclei_Mito_Texture_SumVariance-256-3,Nuclei_Mito_Texture_Variance-256-3
0,C4-1,257,0.999153,2.982219,0.744462,0.006378,0.003888,0.007920,-0.539880,0.076139,0.999619,0.089376,0.007256,20.734105,5.929081
1,C4-1,514,0.998938,1.483146,0.709325,0.007962,0.003887,0.009731,-0.504534,0.080329,0.999521,0.065135,0.008947,8.845046,2.582048
2,C4-1,771,0.999242,4.097950,0.628331,0.005974,0.003888,0.007196,-0.436480,0.062541,0.999648,0.079621,0.006622,18.895394,5.748336
3,C4-1,1028,0.998070,7.148888,0.753846,0.013514,0.003884,0.016842,-0.533855,0.109361,0.999133,0.209727,0.015415,50.066088,14.303744
4,C4-1,1285,0.997498,8.288615,0.793154,0.016905,0.003882,0.021846,-0.563516,0.129575,0.998877,0.286350,0.019418,70.713575,19.750548


In [7]:
stop_profiling(
    start_time=start_time,
    start_mem=start_mem,
    feature_type="Texture",
    well_fov=well_fov,
    patient_id=patient,
    channel=channel,
    compartment=compartment,
    CPU_GPU="CPU",
    output_file_dir=pathlib.Path(
        f"{root_dir}/data/{patient}/extracted_features/run_stats/{well_fov}_{channel}_{compartment}_Texture_CPU.parquet"
    ),
)


        Memory and time profiling for the run:
        Patient ID: NF0014_T1
        Well and FOV: C4-1
        Feature type: Texture
        CPU/GPU: CPU
        Peak memory (tracemalloc): 2539.95 MB
        Current memory (tracemalloc): 407.67 MB
        RSS at end: 583.24 MB
        Time elapsed:
        --- 391.64 seconds ---
        --- 6.53 minutes ---
        --- 0.11 hours ---
    


True